# Figuras para el artículo — FDS Sector Pinturas Colombia
## Cumplimiento SGA · Análisis Exploratorio · n = 50 FDS · 2016–2020

Notebook de producción de figuras para el artículo de investigación.  
Cada figura está configurada para publicación científica (DPI 300, estilo limpio, tipografía consistente).

| Figura | Contenido | Sección del artículo |
|:---:|---|---|
| Fig. 1 | Distribución de categorías de calidad por sección SGA | Resultados — calidad global |
| Fig. 2 | Perfil de calidad por sección según tipo de uso | Resultados — análisis por uso |
| Fig. 3 | Brecha completitud–calidad por sección (cuadrantes) | Resultados — análisis de brecha |
| Fig. 4 | Heatmap de perfiles de calidad por cluster de FDS | Resultados — tipología de FDS |
| Fig. 5 | Comparación de scores globales por tipo de uso | Resultados — análisis estadístico |


---
## Configuración global y carga de datos

In [14]:
import warnings
import re
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist
from pathlib import Path

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

# ── Parámetros de publicación ────────────────────────────────────────────────
DPI_SCREEN  = 130          # previsualización en notebook
DPI_EXPORT  = 300          # exportación a PNG
RAIZ_REPO   = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
FIG_DIR     = RAIZ_REPO / "outputs" / "figuras_articulo"
FIG_DIR.mkdir(parents=True, exist_ok=True)

FACECOLOR   = "#FAFAFA"
GRIDCOLOR   = "#DDDDDD"

# Tipografía
FS_TITLE    = 13
FS_LABEL    = 11
FS_TICK     = 9
FS_ANNOT    = 8

# Paleta semántica (consistente con EDA.ipynb)
C_CONF      = "#2ca02c"    # verde  — CONFIABLE
C_RESTR     = "#ff7f0e"    # naranja — CONFI_RESTR
C_NOCONF    = "#d62728"    # rojo   — NO_CONFIABLE

C_IND       = "#1f77b4"    # azul   — INDUSTRIAL
C_DOM       = "#ff7f0e"    # naranja — DOMÉSTICA

# Estilo global matplotlib
plt.rcParams.update({
    "figure.facecolor"   : FACECOLOR,
    "axes.facecolor"     : FACECOLOR,
    "axes.grid"          : True,
    "grid.color"         : GRIDCOLOR,
    "grid.linewidth"     : 0.6,
    "axes.spines.top"    : False,
    "axes.spines.right"  : False,
    "font.family"        : "sans-serif",
    "font.size"          : FS_TICK,
    "axes.titlesize"     : FS_TITLE,
    "axes.labelsize"     : FS_LABEL,
    "xtick.labelsize"    : FS_TICK,
    "ytick.labelsize"    : FS_TICK,
    "legend.fontsize"    : FS_TICK,
    "legend.framealpha"  : 0.85,
    "legend.edgecolor"   : "#CCCCCC",
    "savefig.dpi"        : DPI_EXPORT,
    "savefig.bbox"       : "tight",
    "savefig.facecolor"  : "white",
})

SECCIONES = {
    1:"Identificación", 2:"Peligros", 3:"Composición",
    4:"Primeros auxilios", 5:"Incendios", 6:"Vertido accidental",
    7:"Manipulación", 8:"Controles/EPP", 9:"Prop. fisicoquímicas",
    10:"Estabilidad", 11:"Toxicología", 12:"Ecotoxicología",
    13:"Eliminación", 14:"Transporte", 15:"Reglamentación", 16:"Otras"
}

print("✅ Configuración global lista · figuras se guardarán en:", FIG_DIR)


✅ Configuración global lista · figuras se guardarán en: figuras_articulo


In [15]:
# Carga y preparación del dataset (idéntica a EDA.ipynb)
df_raw = pd.read_csv("../data/processed/eval.csv", sep=",", encoding="utf-8")

# Eliminar columnas 3.1 (sustancia específica)
drop_cols = [c for c in df_raw.columns if re.search(r'[A-Z]+_3_1_', c)]
df = df_raw.drop(columns=drop_cols).copy()

# Clasificar columnas
meta_cols    = ["FDS","ELABORADA","ID","FABRICANTE","USO"]
item_cols    = [c for c in df.columns if c.startswith("ITEM_")]
cal_cols     = [c for c in df.columns if c.startswith("CALIDAD_")]

# Mapa de calidad ordinal (NO_APLICA → NaN, excluido del denominador)
MAP_Q = {"NO_CONFIABLE": 0, "CONFI_RESTR": 1, "CONFIABLE": 2, "NO_APLICA": np.nan}

# Score por sección (0–100)
SECTIONS = sorted({int(re.search(r'CALIDAD_(\d+)_', c).group(1))
                   for c in cal_cols if re.search(r'CALIDAD_(\d+)_', c)})

sec_score_cols = []
for s in SECTIONS:
    cols_s = [c for c in cal_cols if re.match(rf'CALIDAD_{s}_', c)]
    num = df[cols_s].replace(MAP_Q).apply(pd.to_numeric, errors="coerce")
    df[f"SCORE_SEC_{s}_0_100"] = num.mean(axis=1, skipna=True) * 50
    sec_score_cols.append(f"SCORE_SEC_{s}_0_100")

sec_nums = [int(re.search(r'SCORE_SEC_(\d+)', c).group(1)) for c in sec_score_cols]

# Score global
df["SCORE_GLOBAL_0_100"] = df[sec_score_cols].mean(axis=1, skipna=True)

# Completitud por sección (excluyendo NO_APLICA del denominador)
ANIO_COL = "ELABORADA" if "ELABORADA" in df.columns else None
if ANIO_COL:
    df["ANIO"] = pd.to_numeric(df[ANIO_COL], errors="coerce")

print(f"✅ Dataset: {len(df)} FDS · {len(sec_score_cols)} secciones · "
      f"{df['FABRICANTE'].nunique()} fabricantes · {df['USO'].nunique()} usos")


✅ Dataset: 50 FDS · 16 secciones · 8 fabricantes · 2 usos


---
## Figura 1 — Distribución de categorías de calidad por sección SGA

Cada barra representa el 100% de los ítems calificados de esa sección (se excluyen ítems NO_APLICA del denominador).
Las secciones están ordenadas por score de calidad ascendente para facilitar la lectura de las secciones más críticas.


In [16]:
# ── Figura 1: Distribución de categorías de calidad por sección SGA ─────────
plt.close("all")

# Calcular proporciones por sección
rows = []
for s in SECTIONS:
    cols_s = [c for c in cal_cols if re.match(rf'CALIDAD_{s}_', c)]
    vals = df[cols_s].values.flatten()
    vals = [v for v in vals if v != "NO_APLICA" and pd.notna(v)]
    total = len(vals)
    if total == 0:
        continue
    confiable  = sum(1 for v in vals if v == "CONFIABLE")   / total * 100
    restr      = sum(1 for v in vals if v == "CONFI_RESTR")  / total * 100
    noconf     = sum(1 for v in vals if v == "NO_CONFIABLE") / total * 100
    score      = confiable * 1 + restr * 0.5  # para ordenar
    rows.append({"Sección": s, "Nombre": SECCIONES[s],
                 "CONFIABLE": confiable, "CONFI_RESTR": restr,
                 "NO_CONFIABLE": noconf, "score_ord": score})

df_cat = pd.DataFrame(rows).sort_values("score_ord")

# Etiquetas del eje Y
ylabels = [f"S{int(r.Sección):02d} · {r.Nombre}" for _, r in df_cat.iterrows()]

fig, ax = plt.subplots(figsize=(11, 7), dpi=DPI_SCREEN, facecolor="white")

y = np.arange(len(df_cat))
h = 0.62

# Barras apiladas
b_nc  = ax.barh(y, df_cat["NO_CONFIABLE"], height=h, color=C_NOCONF, label="No confiable")
b_cr  = ax.barh(y, df_cat["CONFI_RESTR"],  height=h, color=C_RESTR,  label="Confiable con restricciones",
                left=df_cat["NO_CONFIABLE"].values)
b_c   = ax.barh(y, df_cat["CONFIABLE"],    height=h, color=C_CONF,   label="Confiable",
                left=(df_cat["NO_CONFIABLE"] + df_cat["CONFI_RESTR"]).values)

# Anotaciones de valor (solo si ≥ 8%)
for i, row in df_cat.reset_index(drop=True).iterrows():
    if row["NO_CONFIABLE"] >= 8:
        ax.text(row["NO_CONFIABLE"]/2, i, f"{row['NO_CONFIABLE']:.0f}%",
                va="center", ha="center", fontsize=7, color="white", fontweight="bold")
    left_cr = row["NO_CONFIABLE"] + row["CONFI_RESTR"]/2
    if row["CONFI_RESTR"] >= 8:
        ax.text(left_cr, i, f"{row['CONFI_RESTR']:.0f}%",
                va="center", ha="center", fontsize=7, color="white", fontweight="bold")
    left_c = row["NO_CONFIABLE"] + row["CONFI_RESTR"] + row["CONFIABLE"]/2
    if row["CONFIABLE"] >= 8:
        ax.text(left_c, i, f"{row['CONFIABLE']:.0f}%",
                va="center", ha="center", fontsize=7, color="white", fontweight="bold")

# Línea de referencia 50%
ax.axvline(50, color="gray", linestyle="--", linewidth=1, alpha=0.7, zorder=0)
ax.text(50.8, len(df_cat)-0.5, "50%", fontsize=8, color="gray")

# Secciones críticas SGA (S2, S4, S8, S11)
crits = {2, 4, 8, 11}
for i, row in df_cat.reset_index(drop=True).iterrows():
    if int(row["Sección"]) in crits:
        ax.get_yticklabels()  # placeholder; mark with star in label
        ax.annotate("★", xy=(101, i), fontsize=9, color="#8B0000",
                    va="center", annotation_clip=False)

ax.set_yticks(y)
ax.set_yticklabels(ylabels, fontsize=FS_TICK)
ax.set_xlim(0, 103)
ax.set_xlabel("Porcentaje de ítems calificados (%)", fontsize=FS_LABEL)
ax.set_title(
    "Distribución de categorías de calidad por sección SGA\n"
    "(ítems NO APLICA excluidos del denominador; ★ = sección crítica para salud)",
    fontsize=FS_TITLE, pad=12
)
ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, -0.07),
    ncol=3,
    fontsize=FS_TICK,
    framealpha=0.9
)

fig.subplots_adjust(bottom=0.16)
ax.grid(axis="x", alpha=0.4)
ax.grid(axis="y", alpha=0)

plt.tight_layout()
outpath = FIG_DIR / "fig1_categorias_por_seccion.png"
plt.savefig(outpath, dpi=DPI_EXPORT, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ Figura 1 guardada → {outpath}")


✅ Figura 1 guardada → figuras_articulo/fig1_categorias_por_seccion.png


---
## Figura 2 — Perfil de calidad por sección según tipo de uso

Comparación del score promedio de calidad (0–100) por sección entre FDS de uso INDUSTRIAL y DOMÉSTICO.
Las líneas verticales punteadas marcan las secciones de mayor criticidad para la seguridad (S2, S4, S8, S11).
El área sombreada entre las curvas visualiza la magnitud de la brecha.


In [17]:
# ── Figura 2: Perfil de calidad por sección según USO ──────────────────────
plt.close("all")

usos = ["INDUSTRIAL", "DOMESTICA"]
pal_uso = {"INDUSTRIAL": C_IND, "DOMESTICA": C_DOM}
labels_uso = {"INDUSTRIAL": "Industrial", "DOMESTICA": "Doméstica"}
crits_sec = [2, 4, 8, 11]

scores_by_uso = {
    uso: df[df["USO"] == uso][sec_score_cols].mean().values
    for uso in usos
}

fig, ax = plt.subplots(figsize=(12, 5.5), dpi=DPI_SCREEN, facecolor="white")

# Área entre curvas (brecha)
ax.fill_between(sec_nums,
                scores_by_uso["INDUSTRIAL"],
                scores_by_uso["DOMESTICA"],
                alpha=0.10, color="gray", zorder=1)

# Líneas por uso
for uso in usos:
    sc = scores_by_uso[uso]
    ax.plot(sec_nums, sc, marker="o", linewidth=2.2,
            color=pal_uso[uso], label=labels_uso[uso],
            markersize=5.5, zorder=3, markeredgewidth=0.5,
            markeredgecolor="white")

# Líneas verticales — secciones críticas
for s_c in crits_sec:
    ax.axvline(s_c, color="#B22222", linestyle=":", linewidth=1.1, alpha=0.65, zorder=2)

# Anotación de la mayor brecha
gaps = np.abs(scores_by_uso["INDUSTRIAL"] - scores_by_uso["DOMESTICA"])
max_gap_idx = int(np.argmax(gaps))
max_gap_sec = sec_nums[max_gap_idx]
y_ind = scores_by_uso["INDUSTRIAL"][max_gap_idx]
y_dom = scores_by_uso["DOMESTICA"][max_gap_idx]
y_mid = (y_ind + y_dom) / 2
_gap_txt = "Brecha max.\n" + "%.1f pts (S%d)" % (gaps[max_gap_idx], max_gap_sec)
ax.annotate(
    _gap_txt,
    xy=(max_gap_sec, y_mid),
    xytext=(max_gap_sec + 1.2, y_mid + 10),
    fontsize=FS_ANNOT, color="dimgray",
    arrowprops=dict(arrowstyle="->", color="dimgray", lw=0.8),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7,
              edgecolor="#CCCCCC")
)

# Etiquetas secciones críticas (parte superior)
for s_c in crits_sec:
    ax.text(s_c, 103, f"S{s_c}", ha="center", va="bottom",
            fontsize=7.5, color="#B22222", fontweight="bold")

# Línea umbral 50
ax.axhline(50, color="gray", linestyle="--", linewidth=0.8,
           alpha=0.5, label="Umbral 50 pts", zorder=1)

ax.set_xticks(sec_nums)
ax.set_xticklabels([f"S{n}" for n in sec_nums], fontsize=FS_TICK)
ax.set_ylim(0, 112)
ax.set_xlim(0.5, 16.5)
ax.set_xlabel("Sección SGA", fontsize=FS_LABEL)
ax.set_ylabel("Score promedio de calidad (0–100)", fontsize=FS_LABEL)
ax.set_title(
    "Perfil de calidad por sección SGA según tipo de uso de la FDS" + chr(10) +
    "(líneas rojas punteadas = secciones críticas S2, S4, S8, S11)",
    fontsize=FS_TITLE, pad=12
)
ax.legend(fontsize=FS_TICK, loc="upper left")

plt.tight_layout()
outpath = FIG_DIR / "fig2_perfil_por_uso.png"
plt.savefig(outpath, dpi=DPI_EXPORT, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ Figura 2 guardada → {outpath}")


✅ Figura 2 guardada → figuras_articulo/fig2_perfil_por_uso.png


---
## Figura 3 — Brecha entre completitud y calidad por sección (análisis de cuadrantes)

Un ítem puede estar *presente* en la FDS pero contener información de baja calidad o confiabilidad.
El eje X mide la completitud (% ítems PRESENTE, sin NO_APLICA); el eje Y mide el score de calidad (0–100).
Las líneas de referencia al 70% dividen el espacio en cuatro cuadrantes de riesgo informacional.


In [18]:
# ── Figura 3: Brecha completitud–calidad (scatter de cuadrantes) ────────────
plt.close("all")

# Calcular completitud y calificación por sección
brecha_rows = []

for s in SECTIONS:

    item_cols_s = [c for c in item_cols if re.match(rf'ITEM_{s}_', c)]
    cal_cols_s  = [c for c in cal_cols  if re.match(rf'CALIDAD_{s}_', c)]

    total_items = 0
    present = 0

    for _, row in df.iterrows():
        for ic in item_cols_s:
            v = row[ic]

            if v == "NO_APLICA" or pd.isna(v):
                continue

            total_items += 1

            if v == "PRESENTE":
                present += 1

    pct_pres = present / total_items * 100 if total_items > 0 else np.nan

    num_s = (
        df[cal_cols_s]
        .replace(MAP_Q)
        .apply(pd.to_numeric, errors="coerce")
    )

    calif_s = num_s.mean(axis=1, skipna=True).mean() * 50

    brecha_rows.append({
        "Sección": s,
        "Nombre": SECCIONES[s],
        "Completitud": pct_pres,
        "Calificacion": calif_s,
        "Brecha": pct_pres - calif_s,
        "N_items": len(item_cols_s)
    })

df_brecha = pd.DataFrame(brecha_rows)

# ── Figura ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(
    figsize=(11, 8.4),
    dpi=DPI_SCREEN,
    facecolor="white"
)

REF = 70

# ── Líneas de referencia ─────────────────────────────────────────────────────
ax.axvline(
    REF,
    color="#666666",
    linestyle="--",
    linewidth=1.2,
    alpha=0.65
)

ax.axhline(
    REF,
    color="#666666",
    linestyle="--",
    linewidth=1.2,
    alpha=0.65
)

ax.text(
    REF + 0.5,
    1,
    f"{REF}%",
    color="#666666",
    fontsize=8
)

ax.text(
    1,
    REF + 1,
    f"{REF} pts",
    color="#666666",
    fontsize=8
)

# ── Fondos de cuadrantes ─────────────────────────────────────────────────────
ax.axvspan(0, REF, ymin=REF/107, ymax=1, color="#FFF9D9", alpha=0.22, zorder=0)
ax.axvspan(REF, 107, ymin=REF/107, ymax=1, color="#DFF2DF", alpha=0.35, zorder=0)
ax.axvspan(0, REF, ymin=0, ymax=REF/107, color="#F2F2F2", alpha=0.32, zorder=0)
ax.fill_between(
    [REF, 105],
    [REF, REF],
    [0, 0],
    color="#FFCCCC",
    alpha=0.30,
    zorder=0
)

# ── Scatter ──────────────────────────────────────────────────────────────────
sizes = (
    df_brecha["N_items"] /
    df_brecha["N_items"].max() * 350 + 80
).values

crits_set = {2, 4, 8, 11}

colors_pts = [
    "#B22222" if int(s) in crits_set else C_IND
    for s in df_brecha["Sección"]
]

ax.scatter(
    df_brecha["Completitud"],
    df_brecha["Calificacion"],
    s=sizes,
    c=colors_pts,
    alpha=0.72,
    edgecolors="white",
    linewidths=0.8,
    zorder=3
)

# ── Etiquetas de secciones ───────────────────────────────────────────────────
for _, row in df_brecha.iterrows():

    sec = int(row["Sección"])

    # Ajustes manuales para evitar solapamientos
    if sec == 10:
        x_off = 3.2
        y_off = -2.2

    elif sec == 14:
        x_off = 1.8
        y_off = 2.2

    elif sec == 3:
        x_off = 1.6
        y_off = 2.0

    elif sec == 6:
        x_off = 1.6
        y_off = -2.0

    elif sec == 5:
        x_off = 1.6
        y_off = 2.4

    elif sec == 7:
        x_off = 1.6
        y_off = 2.4

    elif row["Completitud"] >= 95:
        x_off = -3.8
        y_off = 1.8

    else:
        x_off = 1.6
        y_off = 1.8

    ax.annotate(
        f"S{sec}",
        xy=(row["Completitud"], row["Calificacion"]),
        xytext=(
            row["Completitud"] + x_off,
            row["Calificacion"] + y_off
        ),
        fontsize=8.5,
        fontweight="bold",
        color="#B22222" if sec in crits_set else "dimgray",
        zorder=4
    )

# ── Etiquetas de cuadrantes ──────────────────────────────────────────────────
fs_quad = 10.5

ax.text(
    REF * 0.50,
    97,
    "Low completeness\nHigh quality",
    ha="center",
    va="top",
    fontsize=fs_quad,
    color="#555555",
    style="italic",
    fontweight="bold",
    alpha=0.9
)

ax.text(
    REF + (105 - REF) * 0.50,
    97,
    "High completeness\nHigh quality",
    ha="center",
    va="top",
    fontsize=fs_quad,
    color="#2ca02c",
    style="italic",
    fontweight="bold",
    alpha=0.95
)

ax.text(
    REF * 0.50,
    6,
    "Low completeness\nLow quality",
    ha="center",
    va="bottom",
    fontsize=fs_quad,
    color="#d62728",
    style="italic",
    fontweight="bold",
    alpha=0.95
)

ax.text(
    REF + (105 - REF) * 0.50,
    6,
    "High completeness\nLow quality",
    ha="center",
    va="bottom",
    fontsize=fs_quad,
    color="#B22222",
    style="italic",
    fontweight="bold",
    alpha=0.98
)

# ── Leyenda de tamaños ───────────────────────────────────────────────────────
for n in [2, 4, 7]:

    s_size = n / df_brecha["N_items"].max() * 350 + 80

    ax.scatter(
        [],
        [],
        s=s_size,
        color="gray",
        alpha=0.5,
        label=f"{n} items/section"
    )

# ── Leyenda de colores ───────────────────────────────────────────────────────
ax.scatter([], [], color=C_IND, s=100, label="General section")
ax.scatter([], [], color="#B22222", s=100, label="Health-critical section (S2,S4,S8,S11)")

# ── Configuración general ────────────────────────────────────────────────────
ax.set_xlim(0, 107)
ax.set_ylim(0, 107)

ax.set_xlabel(
    "Average completeness by section (% PRESENT items)",
    fontsize=FS_LABEL
)

ax.set_ylabel(
    "Average quality score by section (0–100)",
    fontsize=FS_LABEL
)

ax.set_title(
    "Quadrant analysis: completeness–quality gap by GHS section\n"
    "Lower-right quadrant = sections with high presence but low-quality content",
    fontsize=FS_TITLE,
    pad=16
)

ax.legend(
    fontsize=7.5,
    loc="upper left",
    ncol=2
)

ax.grid(alpha=0.35)

plt.tight_layout()

outpath = FIG_DIR / "fig3_brecha_cuadrantes.png"

plt.savefig(
    outpath,
    dpi=DPI_EXPORT,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"✅ Figura 3 guardada → {outpath}")

✅ Figura 3 guardada → figuras_articulo/fig3_brecha_cuadrantes.png


---
## Figura 4 — Perfiles de calidad por cluster de FDS (clustering jerárquico Ward)

El clustering Ward sobre los scores seccionales (16 dimensiones) identifica tres tipologías de FDS.
El heatmap muestra el score promedio por sección para cada cluster.


In [19]:
# ── Figura 4: Heatmap de perfiles por cluster ───────────────────────────────
plt.close("all")

# Clustering Ward (3 clusters, idéntico a EDA.ipynb)
X = df[sec_score_cols].fillna(df[sec_score_cols].mean())
Z = linkage(pdist(X, metric="euclidean"), method="ward")
df["CLUSTER"] = fcluster(Z, t=3, criterion="maxclust")

# Reordenar clusters por score medio descendente
cluster_means = df.groupby("CLUSTER")["SCORE_GLOBAL_0_100"].mean()
rank_map = {old: new for new, old in enumerate(
    cluster_means.sort_values(ascending=False).index, start=1)}
df["CLUSTER_R"] = df["CLUSTER"].map(rank_map)

# Perfil por cluster
profile = df.groupby("CLUSTER_R")[sec_score_cols].mean()
profile.columns = ["S" + re.search(r"SCORE_SEC_(\d+)", c).group(1) for c in profile.columns]

# Etiquetas descriptivas para los clusters
cluster_sizes = df.groupby("CLUSTER_R").size()
cluster_med   = df.groupby("CLUSTER_R")["SCORE_GLOBAL_0_100"].median()
row_labels = []
for cl in profile.index:
    n   = cluster_sizes[cl]
    med = cluster_med[cl]
    row_labels.append(f"Cluster {cl}  (n={n}, mediana={med:.0f})")

# Colormap personalizado
cmap_hm = LinearSegmentedColormap.from_list(
    "calidad", ["#d62728", "#ff7f0e", "#ffffcc", "#2ca02c"], N=256)

fig, ax = plt.subplots(figsize=(14, 3.6), dpi=DPI_SCREEN, facecolor="white")
im = ax.imshow(profile.values, aspect="auto", cmap=cmap_hm, vmin=0, vmax=100)

# Anotaciones de valor en cada celda
for i in range(profile.shape[0]):
    for j in range(profile.shape[1]):
        val = profile.values[i, j]
        color_txt = "white" if val < 30 or val > 75 else "black"
        ax.text(j, i, f"{val:.0f}", ha="center", va="center",
                fontsize=8.5, color=color_txt, fontweight="bold")

# Ejes
ax.set_xticks(range(profile.shape[1]))
xticklabs = profile.columns.tolist()
crits_idx = [j for j, col in enumerate(xticklabs) if col in ("S2","S4","S8","S11")]
ax.set_xticklabels(xticklabs, fontsize=FS_TICK)
# Resaltar secciones críticas en el eje X
for j in crits_idx:
    ax.get_xticklabels()[j] if ax.get_xticklabels() else None

ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels(row_labels, fontsize=FS_TICK)

# Marcar secciones críticas con borde rojo
for j in crits_idx:
    for i in range(profile.shape[0]):
        rect = plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                              fill=False, edgecolor="#B22222",
                              linewidth=1.5, zorder=4)
        ax.add_patch(rect)

ax.set_title(
    "Heatmap de perfiles de calidad por sección SGA según cluster de FDS\n"
    "(clustering jerárquico Ward · 3 clusters · bordes rojos = secciones críticas)",
    fontsize=FS_TITLE, pad=12
)

# Colorbar
cbar = plt.colorbar(im, ax=ax, orientation="vertical", pad=0.02, shrink=0.85)
cbar.set_label("Score de calidad (0–100)", fontsize=FS_ANNOT + 1)
cbar.ax.tick_params(labelsize=FS_ANNOT)

# Eliminar bordes del heatmap
for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(bottom=False, left=False)

plt.tight_layout()
outpath = FIG_DIR / "fig4_heatmap_clusters.png"
plt.savefig(outpath, dpi=DPI_EXPORT, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ Figura 4 guardada → {outpath}")


✅ Figura 4 guardada → figuras_articulo/fig4_heatmap_clusters.png


---
## Figura 5 — Comparación de scores globales por tipo de uso (Kruskal-Wallis)

Distribución del score global de calidad (0–100) por tipo de uso.
Se usa Kruskal-Wallis (no ANOVA) por la escala ordinal de los datos y la distribución no normal de los scores.
Los puntos individuales solapados muestran la distribución real de cada FDS.


In [ ]:
# ── Figura 5: Boxplot + strip por USO con Kruskal-Wallis ────────────────────
plt.close("all")

from scipy.stats import kruskal

grupos = [df[df["USO"] == u]["SCORE_GLOBAL_0_100"].dropna().values
          for u in ["INDUSTRIAL", "DOMESTICA"]]
H, p = kruskal(*grupos)
n_ind = len(grupos[0]); n_dom = len(grupos[1])

# Epsilon-squared (tamaño de efecto)
N_total = sum(len(g) for g in grupos)
eps2 = (H - len(grupos) + 1) / (N_total - len(grupos))

fig, ax = plt.subplots(figsize=(7, 6), dpi=DPI_SCREEN, facecolor="white")

uso_labels = ["Industrial", "Doméstica"]
uso_keys   = ["INDUSTRIAL", "DOMESTICA"]
pal_box    = [C_IND, C_DOM]

bp = ax.boxplot(
    [df[df["USO"] == u]["SCORE_GLOBAL_0_100"].dropna().values for u in uso_keys],
    positions=[1, 2], widths=0.45,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.2),
    capprops=dict(linewidth=1.2),
    flierprops=dict(marker="x", markersize=5, alpha=0.4)
)

for patch, color in zip(bp["boxes"], pal_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.45)

# Strip plot (puntos individuales)
rng = np.random.default_rng(42)
for pos, (uso, color) in enumerate(zip(uso_keys, pal_box), start=1):
    vals = df[df["USO"] == uso]["SCORE_GLOBAL_0_100"].dropna().values
    jitter = rng.uniform(-0.08, 0.08, size=len(vals))
    ax.scatter(np.full(len(vals), pos) + jitter, vals,
               color=color, alpha=0.55, s=32, zorder=3,
               edgecolors="white", linewidths=0.4)

# Barra de significancia
y_bar = 97
ax.plot([1, 2], [y_bar, y_bar], color="black", linewidth=1.2)
ax.text(1.5, y_bar + 1.5,
        f"KW: H={H:.2f}, p<0.001\nε²={eps2:.3f} (efecto grande)",
        ha="center", va="bottom", fontsize=8.5,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor="#CCCCCC", alpha=0.9))

# n por grupo
for pos, (uso, n) in enumerate(zip(uso_keys, [n_ind, n_dom]), start=1):
    med = df[df["USO"] == uso]["SCORE_GLOBAL_0_100"].median()
    ax.text(pos, -4, f"n={n}\nMed={med:.1f}", ha="center", va="top",
            fontsize=8.5, color=pal_box[pos-1])

ax.set_xticks([1, 2])
ax.set_xticklabels(uso_labels, fontsize=FS_LABEL)
ax.set_xlim(0.3, 2.7)
ax.set_ylim(-10, 107)
ax.set_ylabel("Calificación global de calidad (0–100)", fontsize=FS_LABEL)
ax.set_title(
    "Distribución del score global de calidad según tipo de uso\n"
    "(Kruskal-Wallis con tamaño de efecto ε²)",
    fontsize=FS_TITLE, pad=12
)
ax.axhline(50, color="gray", linestyle="--", linewidth=0.9,
           alpha=0.5, label="Umbral 50 pts")
ax.legend(fontsize=FS_TICK)

plt.tight_layout()
outpath = FIG_DIR / "fig5_boxplot_uso.png"
plt.savefig(outpath, dpi=DPI_EXPORT, bbox_inches="tight", facecolor="white")
plt.show()
print(f"✅ Figura 5 guardada → {outpath}")


✅ Figura 5 guardada → figuras_articulo/fig5_boxplot_uso.png


---
## Resumen de figuras generadas

| Figura | Archivo | Resolución |
|:---:|---|:---:|
| Fig. 1 | `fig1_categorias_por_seccion.png` | 300 DPI |
| Fig. 2 | `fig2_perfil_por_uso.png` | 300 DPI |
| Fig. 3 | `fig3_brecha_cuadrantes.png` | 300 DPI |
| Fig. 4 | `fig4_heatmap_clusters.png` | 300 DPI |
| Fig. 5 | `fig5_boxplot_uso.png` | 300 DPI |

**Notas de reproducibilidad:**
- Todas las figuras leen el mismo `../data/processed/eval.csv` que `EDA.ipynb`.
- El directorio `figuras_articulo/` se crea automáticamente al ejecutar la celda de configuración.
- Ejecutar con *Restart & Run All* desde el directorio `notebooks/`.


In [21]:
# Verificar que todos los archivos fueron generados
for fig_file in sorted(FIG_DIR.glob("fig*.png")):
    size_kb = fig_file.stat().st_size / 1024
    print(f"  {'✅' if size_kb > 10 else '⚠️ '} {fig_file.name:<42} {size_kb:>7.1f} KB")


  ✅ fig1_categorias_por_seccion.png              318.5 KB
  ✅ fig2_perfil_por_uso.png                      317.5 KB
  ✅ fig3_brecha_cuadrantes.png                   336.8 KB
  ✅ fig4_heatmap_clusters.png                    191.4 KB
  ✅ fig5_boxplot_uso.png                         154.1 KB
  ✅ fig6_scores_fabricante.png                   251.6 KB


# __Figura Combinada__

In [25]:
# ── Figura combinada: A) Calificación global + B) Perfil por sección ─────────
plt.close("all")

from scipy.stats import kruskal
import matplotlib.gridspec as gridspec

# ── Parámetros generales ─────────────────────────────────────────────────────
usos = ["INDUSTRIAL", "DOMESTICA"]

# Paleta similar a la figura original
C_IND_FIG = "#1f77b4"   # azul
C_DOM_FIG = "#6a3d9a"   # morado
C_CRIT    = "#B22222"   # rojo oscuro

pal_uso = {
    "INDUSTRIAL": C_IND_FIG,
    "DOMESTICA": C_DOM_FIG
}

labels_uso = {
    "INDUSTRIAL": "Industrial",
    "DOMESTICA": "Doméstica"
}

crits_sec = [2, 4, 8, 11]  # ★ Secciones críticas para salud

FS_PANEL_TITLE = 11
FS_AXIS = 9.5
FS_TICK_LOCAL = 8
FS_LEGEND = 8
FS_ANNOT_LOCAL = 7.5

# ── Figura ───────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(11.8, 4.6), dpi=DPI_SCREEN, facecolor="white")

gs = gridspec.GridSpec(
    1, 2,
    width_ratios=[1.05, 2.15],
    wspace=0.32
)

axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])

# ══════════════════════════════════════════════════════════════════════════════
# A. Calificación global por tipo de uso
# ══════════════════════════════════════════════════════════════════════════════

grupos = [
    df[df["USO"] == u]["SCORE_GLOBAL_0_100"].dropna().values
    for u in usos
]

H, p = kruskal(*grupos)

n_ind = len(grupos[0])
n_dom = len(grupos[1])

N_total = sum(len(g) for g in grupos)
eps2 = (H - len(grupos) + 1) / (N_total - len(grupos))

uso_labels = ["Industrial", "Doméstica"]
pal_box = [C_IND_FIG, C_DOM_FIG]

# Zonas de interpretación
axA.axhspan(0, 33.33, color="#FDE0DD", alpha=0.45, zorder=0)
axA.axhspan(33.33, 66.67, color="#FFF2CC", alpha=0.45, zorder=0)
axA.axhspan(66.67, 100, color="#DDEEDC", alpha=0.45, zorder=0)

# Líneas de corte
axA.axhline(33.33, color="#E57373", linestyle="--", linewidth=0.8, alpha=0.65)
axA.axhline(66.67, color="#6BAF73", linestyle="--", linewidth=0.8, alpha=0.65)

# Etiquetas de zonas
axA.text(
    2.38, 82,
    "Confiable",
    fontsize=8.5,
    color="#2E8B57",
    ha="right",
    va="center"
)

axA.text(
    2.38, 50,
    "Con\nrestricciones",
    fontsize=8.5,
    color="#D88700",
    ha="right",
    va="center"
)

axA.text(
    2.38, 22,
    "No\nconfiable",
    fontsize=8.5,
    color="#B22222",
    ha="right",
    va="center"
)

# Boxplot
bp = axA.boxplot(
    grupos,
    positions=[1, 2],
    widths=0.50,
    patch_artist=True,
    medianprops=dict(color="black", linewidth=1.6),
    whiskerprops=dict(linewidth=1.0, color="black"),
    capprops=dict(linewidth=1.0, color="black"),
    flierprops=dict(marker="x", markersize=4.5, alpha=0.45)
)

for patch, color in zip(bp["boxes"], pal_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.45)
    patch.set_edgecolor("black")
    patch.set_linewidth(0.9)

# Puntos individuales
rng = np.random.default_rng(42)

for pos, (uso, color) in enumerate(zip(usos, pal_box), start=1):

    vals = df[df["USO"] == uso]["SCORE_GLOBAL_0_100"].dropna().values
    jitter = rng.uniform(-0.07, 0.07, size=len(vals))

    axA.scatter(
        np.full(len(vals), pos) + jitter,
        vals,
        color=color,
        alpha=0.65,
        s=25,
        zorder=3,
        edgecolors="white",
        linewidths=0.35
    )

# Estadístico
y_bar = 98

axA.plot([1, 2], [y_bar, y_bar], color="black", linewidth=1.0)

axA.text(
    1.5,
    y_bar + 1.6,
    f"H={H:.2f}, p<0.001\nε²={eps2:.3f} (efecto grande)",
    ha="center",
    va="bottom",
    fontsize=7.5,
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        edgecolor="#CCCCCC",
        alpha=0.92
    )
)

# n y mediana
for pos, (uso, n) in enumerate(zip(usos, [n_ind, n_dom]), start=1):

    med = df[df["USO"] == uso]["SCORE_GLOBAL_0_100"].median()

    axA.text(
        pos,
        2.5,
        f"n={n}  Med={med:.0f}",
        ha="center",
        va="bottom",
        fontsize=7.5,
        color="dimgray"
    )

axA.set_xticks([1, 2])

axA.set_xticklabels(
    ["Industrial\n(n=25)", "Doméstica\n(n=25)"],
    fontsize=FS_TICK_LOCAL
)

axA.set_xlim(0.35, 2.65)
axA.set_ylim(-6, 112)

axA.set_ylabel(
    "Calificación global de calidad (0–100)",
    fontsize=FS_AXIS
)

axA.set_title(
    "A. Calificación global por tipo de uso",
    fontsize=FS_PANEL_TITLE,
    fontweight="bold",
    pad=18
)

axA.grid(axis="y", alpha=0.25)
axA.grid(axis="x", alpha=0)

axA.spines["top"].set_visible(False)
axA.spines["right"].set_visible(False)

# ══════════════════════════════════════════════════════════════════════════════
# B. Perfil de calidad por sección según tipo de uso
# ══════════════════════════════════════════════════════════════════════════════

scores_by_uso = {
    uso: df[df["USO"] == uso][sec_score_cols].mean().values
    for uso in usos
}

# Sombreado de secciones críticas
for s_c in crits_sec:

    axB.axvspan(
        s_c - 0.42,
        s_c + 0.42,
        color="#F2B6B6",
        alpha=0.18,
        zorder=0
    )

    axB.axvline(
        s_c,
        color=C_CRIT,
        linestyle=":",
        linewidth=1.0,
        alpha=0.65,
        zorder=1
    )

# Área entre curvas
axB.fill_between(
    sec_nums,
    scores_by_uso["INDUSTRIAL"],
    scores_by_uso["DOMESTICA"],
    alpha=0.08,
    color="gray",
    zorder=1
)

# Líneas
for uso in usos:

    sc = scores_by_uso[uso]

    axB.plot(
        sec_nums,
        sc,
        marker="o",
        linewidth=1.8,
        color=pal_uso[uso],
        label=labels_uso[uso],
        markersize=4.5,
        zorder=3,
        markeredgewidth=0.45,
        markeredgecolor="white"
    )

# Umbral 50
axB.axhline(
    50,
    color="gray",
    linestyle="--",
    linewidth=0.8,
    alpha=0.55,
    label="Umbral 50 pts",
    zorder=1
)

# Etiquetas superiores críticas con estrella
for s_c in crits_sec:

    axB.text(
        s_c,
        106,
        f"S{s_c} ★",
        ha="center",
        va="bottom",
        fontsize=7.5,
        color=C_CRIT,
        fontweight="bold"
    )

# Brecha máxima
gaps = np.abs(
    scores_by_uso["INDUSTRIAL"] -
    scores_by_uso["DOMESTICA"]
)

max_gap_idx = int(np.argmax(gaps))
max_gap_sec = sec_nums[max_gap_idx]

y_ind = scores_by_uso["INDUSTRIAL"][max_gap_idx]
y_dom = scores_by_uso["DOMESTICA"][max_gap_idx]
y_mid = (y_ind + y_dom) / 2

gap_txt = f"Brecha máx.\n{gaps[max_gap_idx]:.1f} pts (S{max_gap_sec})"

xytext_gap = (
    (max_gap_sec + 0.75, y_mid + 8)
    if max_gap_sec >= 13
    else (max_gap_sec + 1.1, y_mid + 10)
)

axB.annotate(
    gap_txt,
    xy=(max_gap_sec, y_mid),
    xytext=xytext_gap,
    fontsize=FS_ANNOT_LOCAL,
    color="dimgray",
    ha="left",
    va="center",
    arrowprops=dict(
        arrowstyle="->",
        color="dimgray",
        lw=0.75
    ),
    bbox=dict(
        boxstyle="round,pad=0.25",
        facecolor="white",
        alpha=0.78,
        edgecolor="#CCCCCC"
    )
)

axB.set_xticks(sec_nums)

xtick_labels = axB.set_xticklabels(
    [f"S{n}" for n in sec_nums],
    fontsize=FS_TICK_LOCAL,
    rotation=45,
    ha="right"
)

# Resaltar críticas en eje X
for label, n in zip(xtick_labels, sec_nums):

    if int(n) in crits_sec:
        label.set_color(C_CRIT)
        label.set_fontweight("bold")

axB.set_ylim(0, 112)
axB.set_xlim(0.5, 16.5)

axB.set_xlabel("Sección SGA", fontsize=FS_AXIS)

axB.set_ylabel(
    "Calificación promedio de calidad (0–100)",
    fontsize=FS_AXIS
)

axB.set_title(
    "B. Perfil de calidad por sección según tipo de uso\n"
    "(★ = secciones críticas para salud)",
    fontsize=FS_PANEL_TITLE,
    fontweight="bold",
    pad=14
)

axB.legend(
    fontsize=FS_LEGEND,
    loc="lower right",
    framealpha=0.92
)

axB.grid(axis="both", alpha=0.25)

axB.spines["top"].set_visible(False)
axB.spines["right"].set_visible(False)

# ── Ajuste final ─────────────────────────────────────────────────────────────
plt.subplots_adjust(
    top=0.84,
    bottom=0.20,
    left=0.07,
    right=0.985,
    wspace=0.32
)

outpath = FIG_DIR / "fig_AB_calificacion_uso_perfil_seccion.png"

plt.savefig(
    outpath,
    dpi=DPI_EXPORT,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"✅ Figura combinada guardada → {outpath}")

✅ Figura combinada guardada → figuras_articulo/fig_AB_calificacion_uso_perfil_seccion.png


In [26]:
# ── Figura: Score global de calidad por fabricante ───────────────────────────
plt.close("all")

# Variables esperadas:
# df["FABRICANTE"] o df["FABRICANTE_COD"]
# df["USO"]
# df["SCORE_GLOBAL_0_100"]

fabricante_col = "FABRICANTE" if "FABRICANTE" in df.columns else "FABRICANTE_COD"

# Paleta
C_IND_FIG = "#2E86AB"
C_DOM_FIG = "#E07A5F"

# Resumen por fabricante
plot_df = (
    df.groupby(fabricante_col)
    .agg(
        mean_score=("SCORE_GLOBAL_0_100", "mean"),
        n=("SCORE_GLOBAL_0_100", "count")
    )
    .reset_index()
)

plot_df = plot_df.sort_values("mean_score", ascending=True).reset_index(drop=True)
plot_df["y"] = np.arange(len(plot_df))

# Color de barra según categoría de confiabilidad
def color_barra(x):
    if x < 33.33:
        return "#EF5350"   # No confiable
    elif x < 66.67:
        return "#F9A825"   # Con restricciones
    else:
        return "#4CAF50"   # Confiable

plot_df["bar_color"] = plot_df["mean_score"].apply(color_barra)

fig, ax = plt.subplots(figsize=(11, 6.2), dpi=DPI_SCREEN, facecolor="white")

# Zonas de confiabilidad
ax.axvspan(0, 33.33, color="#FDE0DD", alpha=0.35, zorder=0)
ax.axvspan(33.33, 66.67, color="#FFF2CC", alpha=0.35, zorder=0)
ax.axvspan(66.67, 100, color="#DDEEDC", alpha=0.35, zorder=0)

# Líneas de corte
ax.axvline(33.33, color="#E57373", linestyle="--", linewidth=1.0, alpha=0.75)
ax.axvline(66.67, color="#6BAF73", linestyle="--", linewidth=1.0, alpha=0.75)

# Barras
ax.barh(
    plot_df["y"],
    plot_df["mean_score"],
    color=plot_df["bar_color"],
    alpha=0.82,
    height=0.55,
    zorder=2
)

# Puntos individuales por FDS
rng = np.random.default_rng(42)

for _, row in plot_df.iterrows():

    fabricante = row[fabricante_col]
    y_base = row["y"]

    sub = df[df[fabricante_col] == fabricante].copy()

    for uso, color in {
        "INDUSTRIAL": C_IND_FIG,
        "DOMESTICA": C_DOM_FIG
    }.items():

        vals = sub[sub["USO"] == uso]["SCORE_GLOBAL_0_100"].dropna().values

        if len(vals) == 0:
            continue

        jitter = rng.uniform(-0.16, 0.16, size=len(vals))

        ax.scatter(
            vals,
            np.full(len(vals), y_base) + jitter,
            s=38,
            color=color,
            alpha=0.85,
            edgecolors="white",
            linewidths=0.6,
            zorder=4,
            label=uso
        )

# Etiquetas numéricas de las barras, ajustadas para evitar solapamiento
for _, row in plot_df.iterrows():

    x_txt = row["mean_score"] + 1.6

    if row["mean_score"] >= 85:
        x_txt = row["mean_score"] + 2.4
    elif row["mean_score"] >= 72:
        x_txt = row["mean_score"] + 2.0
    elif row["mean_score"] >= 65:
        x_txt = row["mean_score"] + 1.8

    ax.text(
        x_txt,
        row["y"],
        f"{row['mean_score']:.1f}",
        va="center",
        ha="left",
        fontsize=12,
        fontweight="bold",
        color=row["bar_color"],
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.78,
            pad=0.15
        ),
        zorder=6
    )

# Etiquetas eje Y
ax.set_yticks(plot_df["y"])
ax.set_yticklabels(
    [
        f"{row[fabricante_col]}  (n={int(row['n'])})"
        for _, row in plot_df.iterrows()
    ],
    fontsize=10
)

# Etiquetas de zonas
ax.text(16.5, -0.75, "No confiable", color="#EF5350",
        fontsize=10, fontstyle="italic", ha="center")

ax.text(50, -0.75, "C. restricciones", color="#F9A825",
        fontsize=10, fontstyle="italic", ha="center")

ax.text(83.5, -0.75, "Confiable", color="#4CAF50",
        fontsize=10, fontstyle="italic", ha="center")

# Leyenda sin duplicados
handles, labels = ax.get_legend_handles_labels()
legend_map = {
    "INDUSTRIAL": "Industrial",
    "DOMESTICA": "Doméstica"
}

unique = {}
for h, l in zip(handles, labels):
    if l in legend_map:
        unique[legend_map[l]] = h

ax.legend(
    unique.values(),
    unique.keys(),
    title="Tipo de uso",
    loc="lower right",
    fontsize=10,
    title_fontsize=11,
    framealpha=0.95
)

# Configuración general
ax.set_xlim(0, 105)
ax.set_xlabel("Score global de calidad (0–100)", fontsize=12)
ax.set_title(
    "Score global de calidad por fabricante (FDS del sector pinturas, Colombia)",
    fontsize=14,
    pad=14
)

ax.grid(axis="x", alpha=0.30)
ax.grid(axis="y", alpha=0)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Nota inferior
fig.text(
    0.5,
    0.02,
    "Puntos = FDS individuales. Barras = media. Zonas de color = categoría de confiabilidad SGA.",
    ha="center",
    fontsize=10,
    color="gray",
    style="italic"
)

plt.tight_layout(rect=[0, 0.05, 1, 1])

outpath = FIG_DIR / "fig_score_global_fabricante.png"

plt.savefig(
    outpath,
    dpi=DPI_EXPORT,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print(f"✅ Figura guardada → {outpath}")

✅ Figura guardada → figuras_articulo/fig_score_global_fabricante.png


## Figuras finales del manuscrito después de la conciliación metodológica
Las celdas anteriores son análisis y diseños exploratorios. Las imágenes destinadas a la nueva versión del artículo se regeneran desde la matriz procesada con el script siguiente, en `outputs/bloque_metodologico/figuras/`, sin sobrescribir las figuras exploratorias. La Figura 5 del manuscrito no requiere reemplazo porque su único valor afectado sigue redondeando a 69.

In [ ]:
from pathlib import Path
import runpy
RAIZ_REPO = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
script_final = RAIZ_REPO / 'src' / 'generar_figuras_completitud_actualizadas.py'
assert script_final.exists(), f'No se encuentra el generador: {script_final}'
runpy.run_path(str(script_final), run_name='__main__')
